## pydantic output parser 연습
- 주제 정하기 : 네이버 영화소개 페이지를 긁어와서 제목, 개봉일, 등장인물, 줄거리를 요약하기
- pydantic 으로 출력 양식 정하기

In [ ]:
info = """
극장판 체인소 맨: 레제편상영중
영화
Chainsaw Man The Movie: Reze Arc
2025
문서 저장하기
다음
극장판 체인소 맨: 레제편
개봉
2025.09.24.
등급
15세 이상 관람가
장르
애니메이션, 액션, 모험
국가
일본
러닝타임
100분
배급
소니픽처스코리아
원작
만화
예매하기
 좋아요2,540
정보오류 수정요청
네이버 영화 
정보확인 내용 열고 닫기
다른 사이트 더보기
소개
데블 헌터로 일하는 소년 ‘덴지’는 조직의 배신으로 죽음에 내몰린 순간 전기톱 악마견 ‘포치타’와의 계약으로 하나로 합쳐져 누구도 막을 수 없는 존재 ‘체인소 맨’으로 다시 태어난다. 악마와 사냥꾼, 그리고 정체불명의 적들이 얽힌 잔혹한 전쟁 속에서 ‘레제’라는 이름의 미스터리한 소녀가 ‘덴지’ 앞에 나타나는데… ‘덴지’는 사랑이라는 감정에 이끌려 지금껏 가장 위험한 배틀에 몸을 던진다!
"""

In [2]:
# LangSmith 추적 설정 부분
from dotenv import load_dotenv
import os

load_dotenv()

project_name = "wanted_2nd_prompt_basic"
os.environ["LANGSMITH_PROJECT"] = project_name

In [3]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

#--- 모델 설정 ---#
model = ChatOpenAI(
    temperature=0.1,
    model="gpt-4.1-mini",
    verbose=True
)

In [22]:
from pydantic import BaseModel, Field, ValidationError
from langchain_core.output_parsers import PydanticOutputParser

class MovieSummary(BaseModel):
    title: str
    copyright: str = Field(description="영화를 한줄 소개하는 문구", min_length=10, max_length=40)
    release_date: str
    characters: list[str] = Field(description="등장 인물", min_items=3)
    running_time: int
    summary: str = Field(description="영화 줄거리 요약", min_length=50, max_length=250)

parser = PydanticOutputParser(pydantic_object=MovieSummary)
parser

PydanticOutputParser(pydantic_object=<class '__main__.MovieSummary'>)

In [23]:
fmt = parser.get_format_instructions()
fmt

'The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"title": {"title": "Title", "type": "string"}, "copyright": {"description": "영화를 한줄 소개하는 문구", "maxLength": 40, "minLength": 10, "title": "Copyright", "type": "string"}, "release_date": {"title": "Release Date", "type": "string"}, "characters": {"description": "등장 인물", "items": {"type": "string"}, "minItems": 3, "title": "Characters", "type": "array"}, "running_time": {"title": "Running Time", "type": "integer"}, "summary": {"description": "영화 줄거리 요약", "maxLength": 250, "minLength": 50, "title": "Summary", "type": "strin

In [26]:
prompt = ChatPromptTemplate.from_messages([
    ("system", """영화 소개페이지를 보고 영화정보를 정리해주세요. 
     Json 형식으로 출력해주세요. 
     등장 인물의 수는 3명 이상 뽑아주세요.\n\n
     자세한 양식: {fmt}"""),
     ("user", "다음의 텍스트를 영화 정보로 간단하게 정리\n\n{text}")
]).partial(fmt=fmt)

prompt

ChatPromptTemplate(input_variables=['text'], input_types={}, partial_variables={'fmt': 'The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"title": {"title": "Title", "type": "string"}, "copyright": {"description": "영화를 한줄 소개하는 문구", "maxLength": 40, "minLength": 10, "title": "Copyright", "type": "string"}, "release_date": {"title": "Release Date", "type": "string"}, "characters": {"description": "등장 인물", "items": {"type": "string"}, "minItems": 3, "title": "Characters", "type": "array"}, "running_time": {"title": "Running Time", "type": "integer"}, "summary": {"descripti

In [27]:
chain = prompt | model | parser

chain

ChatPromptTemplate(input_variables=['text'], input_types={}, partial_variables={'fmt': 'The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"title": {"title": "Title", "type": "string"}, "copyright": {"description": "영화를 한줄 소개하는 문구", "maxLength": 40, "minLength": 10, "title": "Copyright", "type": "string"}, "release_date": {"title": "Release Date", "type": "string"}, "characters": {"description": "등장 인물", "items": {"type": "string"}, "minItems": 3, "title": "Characters", "type": "array"}, "running_time": {"title": "Running Time", "type": "integer"}, "summary": {"descripti

In [33]:
try:
    result = chain.invoke({"text": info})
    print(result)
except ValidationError as e:
    print(f"답변을 원하는 형식으로 받아오는 것에 실패했습니다: {e}")

title='Demon Slayer: Kimetsu No Yaiba Infinity Castle Arc' copyright='귀살대와 혈귀의 최종 결전이 펼쳐지는 무한성의 이야기' release_date='2025-08-22' characters=['카마도 탄지로', '아가츠마 젠이츠', '하시비라 이노스케', '염주 렌고쿠 쿄쥬로', '음주 우즈이 텐겐', '토키토 무이치로', '칸로지 미츠리', '키부츠지 무잔'] running_time=155 summary='혈귀로 변한 여동생 네즈코를 구하기 위해 귀살대에 입대한 카마도 탄지로와 동료들은 여러 혈귀와 싸우며 성장한다. 무한열차, 유곽, 도공 마을에서 강력한 혈귀들과 전투를 벌인 후, 귀살대 본부에 나타난 최강의 혈귀 키부츠지 무잔과의 최종 결전을 위해 무한성에서 치열한 싸움을 벌인다.'


In [34]:
print(result.characters)

['카마도 탄지로', '아가츠마 젠이츠', '하시비라 이노스케', '염주 렌고쿠 쿄쥬로', '음주 우즈이 텐겐', '토키토 무이치로', '칸로지 미츠리', '키부츠지 무잔']


In [35]:
from datetime import datetime

result.release_date = datetime.strptime(result.release_date, "%Y-%m-%d").date()
print(result.release_date)

2025-08-22


In [37]:
result.copyright

'귀살대와 혈귀의 최종 결전이 펼쳐지는 무한성의 이야기'